<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.png" width="250"/>

In [ ]:
import xarray as xr
# 1. Abrir com chunks (Dask) permite carregar apenas o necessário da memória
nc_file = '/home/henrique/Documentos/ERA5_merged.nc4'
era5_ds_fast = xr.open_dataset(nc_file)
era5_2015_fast = era5_ds_fast.sel(time='2014')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Coordenadas de Jaboticabal
lat_jab = -21.25
lon_jab = -48.35

print(f"Filtrando dados para Jaboticabal (Lat: {lat_jab}, Lon: {lon_jab})...")

# 1. Selecionando o ponto geográfico
jaboticabal_ds = era5_2015_fast.sel(lat=lat_jab, lon=lon_jab, method='nearest')

# 1. Definir as variáveis para plotar
vars_to_plot = ['TMAX', 'TMIN', 'RH2M', 'WS2M', 'ALL_SKY_SFC_SW_DWN', 'PRECTOTCORR']

# 2. Preparar o DataFrame completo para Jaboticabal
# Extraímos os meses via xarray antes da conversão para evitar o erro de .dt
month_values = jaboticabal_ds.time.dt.month.values

# Convertendo o dataset filtrado para um DataFrame do Pandas
df_all_vars = jaboticabal_ds[vars_to_plot].to_dataframe().reset_index()
df_all_vars['month'] = month_values

# 3. Criar a figura com subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

meses_labels = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

for i, var in enumerate(vars_to_plot):
    sns.boxplot(data=df_all_vars, x='month', y=var, ax=axes[i], palette='viridis', hue='month', legend=False)

    axes[i].set_title(f'Distribuição de {var}', fontsize=12)
    axes[i].set_xlabel('Mês')
    axes[i].set_ylabel(var)
    axes[i].set_xticks(range(12))
    axes[i].set_xticklabels(meses_labels)
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

plt.suptitle('Variáveis Climáticas em Jaboticabal - 2014 (ERA5)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import xarray as xr
# 1. Abrir com chunks (Dask) permite carregar apenas o necessário da memória
nc_file = '/home/henrique/Documentos/CNN_CMCC-ESM2_historico_merged.nc4'
era5_ds_fast = xr.open_dataset(nc_file, chunks={'time': 10})
era5_2015_fast = era5_ds_fast.sel(time='2014')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Coordenadas de Jaboticabal
lat_jab = -21.25
lon_jab = -48.35

print(f"Filtrando dados para Jaboticabal (Lat: {lat_jab}, Lon: {lon_jab})...")

# 1. Selecionando o ponto geográfico
jaboticabal_ds = era5_2015_fast.sel(lat=lat_jab, lon=lon_jab, method='nearest')

# 1. Definir as variáveis para plotar
vars_to_plot = ['TMAX', 'TMIN', 'RH2M', 'WS2M', 'ALL_SKY_SFC_SW_DWN', 'PRECTOTCORR']

# 2. Preparar o DataFrame completo para Jaboticabal
# Extraímos os meses via xarray antes da conversão para evitar o erro de .dt
month_values = jaboticabal_ds.time.dt.month.values

# Convertendo o dataset filtrado para um DataFrame do Pandas
df_all_vars = jaboticabal_ds[vars_to_plot].to_dataframe().reset_index()
df_all_vars['month'] = month_values

# 3. Criar a figura com subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

meses_labels = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

for i, var in enumerate(vars_to_plot):
    sns.boxplot(data=df_all_vars, x='month', y=var, ax=axes[i], palette='viridis', hue='month', legend=False)

    axes[i].set_title(f'Distribuição de {var}', fontsize=12)
    axes[i].set_xlabel('Mês')
    axes[i].set_ylabel(var)
    axes[i].set_xticks(range(12))
    axes[i].set_xticklabels(meses_labels)
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

plt.suptitle('Variáveis Climáticas em Jaboticabal - 2014 (CMIP6 - CNN)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show();

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import cftime

# 1. Caminhos dos arquivos
nc_bruto = "G:\\Drives compartilhados\\GAS2-Henrique\\PROJECOES_ENSEMBLE\\CNN_CMCC\\CNN_CMCC-ESM2_ssp126_merged.nc4"
nc_corr = "G:\\Drives compartilhados\\GAS2-Henrique\\PROJECOES_ENSEMBLE\\LOCI_CMCC-ESM2_ssp126_2015_Corrigido_FULL.nc"

# 2. Carregando os datasets (sem forçar chunks para evitar o warning, já que vamos ler só 1 dia)
ds_bruto = xr.open_dataset(nc_bruto, engine="h5netcdf")
ds_corr = xr.open_dataset(nc_corr, engine="netcdf4")

# 3. Definindo a data compatível com o cftime (DatetimeNoLeap)
ano, mes, dia = 2015, 5, 15
data_teste = cftime.DatetimeNoLeap(ano, mes, dia, 12, 0, 0) # Ajustado para o formato interno do seu arquivo

# 4. Extraindo os dados do dia escolhido
var_bruto = 'PRECTOTCORR' if 'PRECTOTCORR' in ds_bruto.data_vars else 'PRECTOT'

# Usamos .sel() e puxamos para a memória (.compute())
chuva_bruta = ds_bruto[var_bruto].sel(time=data_teste, method='nearest').compute()
chuva_corr = ds_corr['PRECTOTCORR'].sel(time=data_teste, method='nearest').compute()

# 5. Configuração do Plot com Cartopy
fig, axes = plt.subplots(1, 2, figsize=(16, 6), subplot_kw={'projection': ccrs.PlateCarree()})

for ax in axes:
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':')

# Plot Bruto
im1 = chuva_bruta.plot(
    ax=axes[0],
    cmap='Blues',
    vmin=0.1, vmax=30,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
axes[0].set_title(f'Bruto (CNN) - {ano}-{mes:02d}-{dia:02d}\nNote o drizzle effect nas áreas claras')

# Plot Corrigido (LOCI)
im2 = chuva_corr.plot(
    ax=axes[1],
    cmap='Blues',
    vmin=0.1, vmax=30,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
axes[1].set_title(f'Corrigido (LOCI) - {ano}-{mes:02d}-{dia:02d}\nExtremos preservados e drizzle removido')

# Adicionando uma barra de cores compartilhada
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im2, cax=cbar_ax, label='Precipitação (mm/dia)', extend='max')

plt.suptitle('Comparação de Viés Espacial: Ocorrência e Intensidade (LOCI)', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout(rect=[0, 0, 0.9, 1])

plt.show()

# Fechando os datasets para liberar memória
ds_bruto.close()
ds_corr.close()